# Figure 3B (UHVDB r6). Host predictions

Revision of `figure_3b.ipynb` using the released UHVDB r6 metadata table
(`toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz`).

Host lineage and ICTV class are already joined into that table
(`host_lineage`, `ictv_class`), so the precomputed r4 metadata TSV is no longer needed.

Body-site stacked bars exclude `Other` (as before) and also exclude `Blood`.

Eukaryotic viruses: original DNA `EUK_CLASSES`, plus all `Riboviria` except
RNA phage classes (`Leviviricetes`, `Cystoviricetes`).


In [1]:
### Load packages
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

META = Path(
    "/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB"
    "/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz"
)
OUTDIR = Path("plots")
OUTDIR.mkdir(exist_ok=True)


In [2]:
### Load UHVDB r6 metadata (genomovar representatives only)
EUK_CLASSES = [
    "Papovaviricetes",
    "Cardeaviricetes",
    "Pokkesviricetes",
    "Arfiviricetes",
    "Repensiviricetes",
    "Quintoviricetes",
    "Megaviricetes",
    "Naldaviricetes",
    "Herviviricetes",
]
# Riboviria RNA bacteriophage classes (exclude from Eukaryote override)
RNA_PHAGE_CLASSES = [
    "Leviviricetes",
    "Cystoviricetes",
]

meta = pl.read_csv(
    META,
    separator="\t",
    null_values=["NA", ""],
    columns=[
        "uhvdb_id",
        "genomovar_rep",
        "seq_name",
        "seqhash_rep",
        "body_site",
        "host_lineage",
        "ictv_class",
        "genomad_taxonomy",
        "species_rep",
        "genus_cluster_id",
        "family_cluster_id",
    ],
    infer_schema_length=10000,
)

genomovars = meta.filter(
    (pl.col("uhvdb_id") == pl.col("genomovar_rep"))
    & (pl.col("seq_name") == pl.col("seqhash_rep"))
)
print(f"total genomovar reps: {genomovars.height:,}")
print(genomovars.group_by("body_site").len().sort("len", descending=True))


total genomovar reps: 925,479
shape: (6, 2)
┌────────────┬────────┐
│ body_site  ┆ len    │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ Gut        ┆ 532722 │
│ Airways    ┆ 282136 │
│ Other      ┆ 85166  │
│ Blood      ┆ 9539   │
│ Skin       ┆ 8363   │
│ Urogenital ┆ 7553   │
└────────────┴────────┘


In [3]:
### Extract host family from GTDB lineage; label eukaryotic viruses
# Eukaryote if: (1) DNA euk class in EUK_CLASSES, or (2) Riboviria except RNA phage classes.
uhvdb_combinedhost_w_meta = (
    genomovars
    .with_columns(
        pl.col("host_lineage")
        .str.extract(r"(?:^|;)f__([^;]*)", 1)
        .replace("", None)
        .alias("host_family")
    )
    .with_columns(
        (
            pl.col("ictv_class").is_in(EUK_CLASSES)
            | (
                pl.col("genomad_taxonomy").fill_null("").str.contains("Riboviria")
                & ~pl.col("ictv_class").is_in(RNA_PHAGE_CLASSES)
            )
        ).alias("is_eukaryote")
    )
    .with_columns(
        pl.when(pl.col("is_eukaryote"))
        .then(pl.lit("Eukaryote"))
        .otherwise(pl.col("host_family"))
        .alias("final_taxonomy")
    )
    .select([
        "uhvdb_id",
        "body_site",
        "final_taxonomy",
        "species_rep",
        "genus_cluster_id",
        "family_cluster_id",
    ])
)

n_total = uhvdb_combinedhost_w_meta.height
n_with_host = uhvdb_combinedhost_w_meta.filter(pl.col("final_taxonomy").is_not_null()).height
n_euk = uhvdb_combinedhost_w_meta.filter(pl.col("final_taxonomy") == "Eukaryote").height
print(
    f"Genomovars with host family prediction: {n_with_host:,} / {n_total:,} "
    f"({n_with_host / n_total * 100:.2f}%)"
)
print(f"  of which labeled Eukaryote: {n_euk:,}")
print(f"Prop of genomovars with host family prediction: {n_with_host / n_total:.4f}")


Genomovars with host family prediction: 821,080 / 925,479 (88.72%)
  of which labeled Eukaryote: 51,410
Prop of genomovars with host family prediction: 0.8872


In [4]:
### Species / genus / family clusters with ≥1 genomovar host prediction
rank_cols = {
    "species": "species_rep",
    "genus": "genus_cluster_id",
    "family": "family_cluster_id",
}

rows = []
for rank, col in rank_cols.items():
    clusters = uhvdb_combinedhost_w_meta.filter(pl.col(col).is_not_null())
    n_total = clusters.select(pl.col(col).n_unique()).item()
    n_with_host = (
        clusters
        .filter(pl.col("final_taxonomy").is_not_null())
        .select(pl.col(col).n_unique())
        .item()
    )
    pct = n_with_host / n_total * 100 if n_total else float("nan")
    print(
        f"{rank.capitalize()} clusters with ≥1 genomovar host prediction: "
        f"{n_with_host:,} / {n_total:,} ({pct:.2f}%)"
    )
    rows.append(
        {
            "rank": rank,
            "n_with_host_prediction": n_with_host,
            "n_total": n_total,
            "percent": pct,
        }
    )

cluster_host_coverage = pl.DataFrame(rows)
cluster_host_coverage

Species clusters with ≥1 genomovar host prediction: 257,524 / 291,551 (88.33%)


Genus clusters with ≥1 genomovar host prediction: 45,061 / 53,492 (84.24%)
Family clusters with ≥1 genomovar host prediction: 1,302 / 1,590 (81.89%)


rank,n_with_host_prediction,n_total,percent
str,i64,i64,f64
"""species""",257524,291551,88.328972
"""genus""",45061,53492,84.238765
"""family""",1302,1590,81.886792


In [5]:
### Percent of genomovars with host species / genus / family
# Rank presence from GTDB host_lineage (s__/g__/f__). Eukaryote-labeled viruses count at all ranks.
host_rank_df = (
    genomovars
    .with_columns(
        (
            pl.col("ictv_class").is_in(EUK_CLASSES)
            | (
                pl.col("genomad_taxonomy").fill_null("").str.contains("Riboviria")
                & ~pl.col("ictv_class").is_in(RNA_PHAGE_CLASSES)
            )
        ).alias("is_eukaryote"),
        pl.col("host_lineage").str.extract(r"(?:^|;)s__([^;]*)", 1).replace("", None).alias("host_species"),
        pl.col("host_lineage").str.extract(r"(?:^|;)g__([^;]*)", 1).replace("", None).alias("host_genus"),
        pl.col("host_lineage").str.extract(r"(?:^|;)f__([^;]*)", 1).replace("", None).alias("host_family"),
    )
)

n_genomovars = host_rank_df.height
rows = []
for rank, col in [("species", "host_species"), ("genus", "host_genus"), ("family", "host_family")]:
    n_with = host_rank_df.filter(
        pl.col(col).is_not_null() | pl.col("is_eukaryote")
    ).height
    pct = n_with / n_genomovars * 100
    print(
        f"Genomovars with host {rank}: {n_with:,} / {n_genomovars:,} ({pct:.2f}%)"
    )
    rows.append(
        {
            "host_rank": rank,
            "n_with_host": n_with,
            "n_genomovars": n_genomovars,
            "percent": pct,
        }
    )

genomovar_host_rank_coverage = pl.DataFrame(rows)
genomovar_host_rank_coverage


Genomovars with host species: 483,097 / 925,479 (52.20%)
Genomovars with host genus: 760,377 / 925,479 (82.16%)
Genomovars with host family: 821,080 / 925,479 (88.72%)


host_rank,n_with_host,n_genomovars,percent
str,i64,i64,f64
"""species""",483097,925479,52.199672
"""genus""",760377,925479,82.160373
"""family""",821080,925479,88.719463


In [6]:
### Get top 10 most prevalent host families overall
# Exclude Other and Blood from body-site analyses.
SITE_ORDER = ["Airways", "Gut", "Skin", "Urogenital"]

top_10_counts = (
    uhvdb_combinedhost_w_meta
    .filter(
        ~pl.col("body_site").is_in(["Other", "Blood"])
        & pl.col("final_taxonomy").is_not_null()
    )
    .group_by("final_taxonomy")
    .len()
    .sort("len", descending=True)
    .head(10)
)
top_10_families = top_10_counts["final_taxonomy"].to_list()

print("Top 10 host families overall:")
print(top_10_counts)

host_family_classified = (
    uhvdb_combinedhost_w_meta
    .filter(~pl.col("body_site").is_in(["Other", "Blood"]))
    .with_columns(
        pl.when(pl.col("final_taxonomy").is_in(top_10_families))
        .then(pl.col("final_taxonomy"))
        .when(pl.col("final_taxonomy").is_null())
        .then(pl.lit("No host prediction"))
        .otherwise(pl.lit("Other"))
        .alias("family_classified")
    )
)

uhvdb_host_annotations = (
    host_family_classified
    .group_by(["body_site", "family_classified"])
    .len()
)
uhvdb_host_annotations


Top 10 host families overall:
shape: (10, 2)
┌────────────────────┬────────┐
│ final_taxonomy     ┆ len    │
│ ---                ┆ ---    │
│ str                ┆ u32    │
╞════════════════════╪════════╡
│ Bacteroidaceae     ┆ 112778 │
│ Lachnospiraceae    ┆ 99403  │
│ Ruminococcaceae    ┆ 54867  │
│ Streptococcaceae   ┆ 52041  │
│ Oscillospiraceae   ┆ 36009  │
│ Veillonellaceae    ┆ 33767  │
│ Actinomycetaceae   ┆ 31740  │
│ Neisseriaceae      ┆ 23677  │
│ Enterobacteriaceae ┆ 21083  │
│ Eukaryote          ┆ 19756  │
└────────────────────┴────────┘


body_site,family_classified,len
str,str,u32
"""Gut""","""Lachnospiraceae""",93266
"""Gut""","""No host prediction""",74400
"""Airways""","""Bacteroidaceae""",20949
"""Airways""","""Streptococcaceae""",45592
"""Airways""","""Ruminococcaceae""",67
…,…,…
"""Skin""","""Other""",4040
"""Skin""","""Eukaryote""",2602
"""Urogenital""","""Streptococcaceae""",221


In [7]:
### Top 5 host families by body site
# Percentages are of all genomovars at that body site (Other / Blood excluded).
SITE_ORDER = ["Airways", "Gut", "Skin", "Urogenital"]

for site in SITE_ORDER:
    site_df = uhvdb_combinedhost_w_meta.filter(pl.col("body_site") == site)
    n_site = site_df.height
    top5 = (
        site_df
        .filter(pl.col("final_taxonomy").is_not_null())
        .group_by("final_taxonomy")
        .len()
        .sort("len", descending=True)
        .head(5)
        .with_columns((pl.col("len") / n_site * 100).alias("percent"))
    )
    print(f"\n{site} (n={n_site:,} genomovars)")
    for row in top5.iter_rows(named=True):
        print(
            f"  {row['final_taxonomy']}: {row['len']:,} ({row['percent']:.2f}%)"
        )



Airways (n=282,136 genomovars)
  Streptococcaceae: 45,592 (16.16%)
  Actinomycetaceae: 29,837 (10.58%)
  Veillonellaceae: 27,692 (9.82%)
  Neisseriaceae: 22,870 (8.11%)
  Bacteroidaceae: 20,949 (7.43%)



Gut (n=532,722 genomovars)
  Lachnospiraceae: 93,266 (17.51%)
  Bacteroidaceae: 91,226 (17.12%)
  Ruminococcaceae: 54,776 (10.28%)
  Oscillospiraceae: 35,966 (6.75%)
  Enterobacteriaceae: 18,733 (3.52%)

Skin (n=8,363 genomovars)
  Eukaryote: 2,602 (31.11%)
  Staphylococcaceae: 992 (11.86%)
  Mycobacteriaceae: 873 (10.44%)
  Propionibacteriaceae: 664 (7.94%)
  Pseudomonadaceae: 294 (3.52%)

Urogenital (n=7,553 genomovars)
  Lactobacillaceae: 1,836 (24.31%)
  Bifidobacteriaceae: 762 (10.09%)
  Eukaryote: 555 (7.35%)
  Bacteroidaceae: 537 (7.11%)
  Enterobacteriaceae: 533 (7.06%)


In [8]:
# Create stacked bar chart showing host family distribution by body site
SITE_ORDER = ["Airways", "Gut", "Skin", "Urogenital"]

family_by_site = (
    uhvdb_host_annotations
    .pivot(index="body_site", on="family_classified", values="len", aggregate_function="sum")
    .fill_null(0)
)

family_by_site_pd = family_by_site.to_pandas().set_index("body_site")
family_by_site_pd = family_by_site_pd.reindex(
    [s for s in SITE_ORDER if s in family_by_site_pd.index]
)

totals = family_by_site_pd.sum(axis=1)
family_proportions = family_by_site_pd.div(totals, axis=0) * 100

special_labels = {"No host prediction", "Other"}
family_avg = family_proportions.mean().sort_values(ascending=False)
named = [c for c in family_avg.index if c not in special_labels]
ordered_cols = (
    family_avg.loc[named].sort_values(ascending=False).index.tolist()
    + [c for c in ["Other", "No host prediction"] if c in family_proportions.columns]
)
family_proportions = family_proportions[ordered_cols]

plt.rcParams.update({"font.size": 14})
n_families = len([c for c in family_proportions.columns if c not in special_labels])
base_colors = plt.cm.tab20.colors[:n_families]
base_color_iter = iter(base_colors)
colors = []
for col in family_proportions.columns:
    if col == "No host prediction":
        colors.append("#D3D3D3")
    elif col == "Other":
        colors.append("#808080")
    else:
        colors.append(next(base_color_iter))

ax = family_proportions.plot(kind="bar", stacked=True, figsize=(12, 6), color=colors)
plt.xticks(rotation=0)
plt.xlabel("Body site", fontdict={"fontweight": "bold"})
plt.ylabel("Percentage of genomovars", fontdict={"fontweight": "bold"})
plt.legend(title="Predicted host", bbox_to_anchor=(1.0, 1), loc="upper left")

for i, (idx, total) in enumerate(totals.items()):
    ax.text(i, 102, f"n={int(total):,}", ha="center", va="bottom", fontsize=12)

plt.ylim(0, 115)
plt.tight_layout()
plt.savefig(OUTDIR / "figure_3b_host_by_body_site_r6.png", dpi=300, bbox_inches="tight")
family_proportions.to_csv(OUTDIR / "figure_3b_host_by_body_site_r6.tsv", sep="\t")
plt.show()


In [9]:
family_proportions

,Eukaryote,Bacteroidaceae,Streptococcaceae,Lachnospiraceae,Actinomycetaceae,Enterobacteriaceae,Veillonellaceae,Ruminococcaceae,Neisseriaceae,Oscillospiraceae,Other,No host prediction
body_site,,,,,,,,,,,,
Airways,1.887742,7.425142,16.159583,2.130887,10.575396,0.589786,9.815125,0.023747,8.106020,0.007089,34.940950,8.338532
Gut,2.116113,17.124504,1.123288,17.507443,0.283074,3.516468,1.128356,10.282286,0.142476,6.751364,26.058620,13.966009
Skin,31.113237,0.789190,2.917613,0.227191,2.295827,1.829487,0.298936,0.131532,0.406553,0.191319,48.308023,11.491092
Urogenital,7.348074,7.109758,2.925990,1.403416,2.687674,7.056799,0.516351,0.172117,0.185357,0.092678,60.518999,9.982788
